# Task: Build a Campus FAQ Chatbot using RAG

## Objective
Learn how Retrieval-Augmented Generation (RAG) works by building a small chatbot that answers
questions about a college. You will turn FAQ entries into vectors, store them in a mini vector
database (FAISS), retrieve the most relevant ones for a question, and have an LLM write the answer
from them.

## Why RAG?
A general-purpose LLM has never seen your college's library hours or parking rules. Asked anyway,
it will often produce a confident, plausible and wrong answer. RAG fixes this by **finding the
relevant facts first** and **handing them to the model** with the question, so the answer comes from
your documents rather than the model's memory.

## What you will build

```
Question ──▶ Embed ──▶ Search FAISS ──▶ Top 3 FAQ entries ──▶ Prompt ──▶ LLM ──▶ Answer
```

## A concrete example
A student types: **"Can I still study in the library at 11 at night during exams?"**

1. **Retrieve.** The question shares almost no words with the FAQ entry *"When is the library open?"*,
   but its meaning is close, so a vector search ranks that entry first.
2. **Augment.** The entry's text (open until midnight during exam weeks) goes into the prompt with the question.
3. **Generate.** The LLM answers: *"Yes. During exam weeks the library stays open until midnight."*

If the same student asks **"Who won the hockey game last night?"**, nothing in the FAQ is relevant. A good
RAG system notices that and says it does not know, instead of inventing an answer. You will build both behaviours.

**Time:** about 40 minutes. Work through the steps in order; each ends with a checkpoint.

## Step 0: Setup

Install the packages. `%pip` installs into the notebook's own Python environment.

In [1]:
%pip install -q sentence-transformers faiss-cpu numpy transformers torch streamlit

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.12.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 2.3.0 which is incompatible.
gradio-client 2.4.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.3.0 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1: Prepare the data

Each entry is a question and its answer. In a real system this would come from the college website, a
handbook or a help-desk knowledge base.

The data below is for a **fictional college**, so the details are made up. Replace or extend it with real
information about your own campus if you like.

**Task:** run the cell, then add **two more** entries of your own at the end of the list.

In [2]:
faqs = [
    {"q": "When is the library open?",
     "a": "The main library is open 8 AM to 10 PM Monday to Thursday, 8 AM to 6 PM on Friday, and 10 AM to 4 PM on weekends. During exam weeks it stays open until midnight."},
    {"q": "How do I book a study room?",
     "a": "Study rooms are booked through the library portal up to 7 days in advance. Each booking is 2 hours, and you can hold one booking at a time."},
    {"q": "Where do I get my student ID card?",
     "a": "Student ID cards are issued at the Student Services desk in Building A, room 104. Bring a government photo ID and your acceptance letter."},
    {"q": "How do I connect to campus Wi-Fi?",
     "a": "Connect to the network called NB-Secure and sign in with your student email and password. Guests can use NB-Guest, which needs no password but is slower."},
    {"q": "What should I do if I forget my password?",
     "a": "Reset it at the IT self-service portal using your recovery email. If that fails, visit the IT Help Desk in Building C, room 210."},
    {"q": "Where can I print?",
     "a": "Printers are in the library and in Building C. Black-and-white pages cost 10 cents and colour pages cost 50 cents, charged to your student card."},
    {"q": "Is there free tutoring?",
     "a": "Yes. The Learning Centre offers free drop-in tutoring for math, writing and programming, Monday to Friday from 10 AM to 4 PM in Building B, room 120."},
    {"q": "How do I see a counsellor?",
     "a": "Counselling is free and confidential. Book an appointment online through Student Wellness or call the front desk. Same-day crisis appointments are available."},
    {"q": "Where can I park?",
     "a": "Students park in Lots 3 and 4 with a semester permit, which costs $180 and is bought online. Lots 1 and 2 are for staff and visitors."},
    {"q": "What is the deadline to drop a course without a failing grade?",
     "a": "You can drop a course without academic penalty until the end of week 8. After that, the course shows as a failing grade on your transcript."},
    {"q": "Does the cafeteria have vegetarian options?",
     "a": "Yes. The main cafeteria in Building A has a vegetarian and vegan station every weekday from 11 AM to 3 PM, and halal options are labelled at every counter."},
    {"q": "How do I contact Accessibility Services?",
     "a": "Email Accessibility Services or visit Building A, room 112. Register early in the term so accommodations such as extra exam time are in place before your first test."},
]

print(len(faqs), "FAQ entries")
faqs[0]

12 FAQ entries


{'q': 'When is the library open?',
 'a': 'The main library is open 8 AM to 10 PM Monday to Thursday, 8 AM to 6 PM on Friday, and 10 AM to 4 PM on weekends. During exam weeks it stays open until midnight.'}

**Checkpoint:** you should see `14 FAQ entries` after adding your two.

## Step 2: Split the data into chunks

A **chunk** is the unit you search over. Chunk choice matters more than almost anything else in RAG.

A tempting shortcut is to split a text file line by line. With FAQ data that puts each **question on one
line and its answer on another**, so they become separate chunks. A user's question is most similar to the
stored *question* line, so the search returns the question back instead of the answer.

The fix is to keep each question **together with** its answer: **one Q&A pair = one chunk.**

In [3]:
chunks = [f"Q: {item['q']}\nA: {item['a']}" for item in faqs]

print(len(chunks), "chunks\n")
print(chunks[0])

12 chunks

Q: When is the library open?
A: The main library is open 8 AM to 10 PM Monday to Thursday, 8 AM to 6 PM on Friday, and 10 AM to 4 PM on weekends. During exam weeks it stays open until midnight.


**Checkpoint:** each chunk should contain both a `Q:` and an `A:` line.

## Step 3: Create embeddings

An **embedding** is a list of numbers (a vector) that captures the meaning of a piece of text. Texts with
similar meaning get vectors that point in similar directions, even when they share no words.

`all-MiniLM-L6-v2` is a small, fast model that turns any text into a 384-number vector. The first run
downloads it (about 90 MB).

`normalize_embeddings=True` scales every vector to length 1. Then the dot product of two vectors equals
their **cosine similarity**: 1.0 means identical meaning, values near 0 mean unrelated.

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks, normalize_embeddings=True).astype("float32")
print("shape:", embeddings.shape)   # (number of chunks, 384)

c:\Users\ulviyya2025!\Desktop\Lambton College\Term 2 Subjects\Advanced Python AI & ML Tools\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1025 18:55:03.616000 44060 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17152.23it/s]


shape: (12, 384)


**See it for yourself.** Compare a question with two sentences: one that means the same thing in different words, one that is unrelated.

In [5]:
a, b, c = model.encode(
    ["When is the library open?",
     "What are the library's hours?",
     "How much does a parking permit cost?"],
    normalize_embeddings=True,
)
print("same meaning :", round(float(a @ b), 2))
print("unrelated    :", round(float(a @ c), 2))

same meaning : 0.71
unrelated    : 0.15


**Checkpoint:** the first score should be clearly higher than the second, even though the two library sentences share only one word.

## Step 4: Build the FAISS index

FAISS is a library for fast vector search. Think of the index as a mini vector database held in memory.

`IndexFlatIP` compares the question against every stored vector using the inner (dot) product. Because the
vectors are normalized, that is cosine similarity, and a **higher score means more relevant**. "Flat" means it
checks every vector exactly, which is perfect for small data. Large systems use approximate indexes to
search millions of vectors quickly.

In [6]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("vectors in index:", index.ntotal)

vectors in index: 12


**Checkpoint:** `vectors in index` should equal your number of chunks.

## Step 5: Retrieve

Embed the user's question with **the same model**, then ask FAISS for the `k` most similar chunks.
Looking at the top 3, rather than only the best match, shows you what the search actually found and
gives the LLM more context to work with.

In [7]:
def retrieve(question, k=3):
    q_emb = model.encode([question], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_emb, k)
    return [(float(s), chunks[i]) for s, i in zip(scores[0], ids[0])]

for score, chunk in retrieve("Can I still study in the library at 11 at night during exams?"):
    print(f"{score:.2f}  {chunk.splitlines()[0]}")

0.55  Q: When is the library open?
0.55  Q: How do I book a study room?
0.39  Q: How do I contact Accessibility Services?


Now try a question the FAQ cannot answer. FAISS **always** returns k results, however unrelated, so
the scores are what tell you whether anything relevant was found.

In [8]:
for score, chunk in retrieve("Who won the hockey game last night?"):
    print(f"{score:.2f}  {chunk.splitlines()[0]}")

0.04  Q: When is the library open?
0.02  Q: What should I do if I forget my password?
0.00  Q: Is there free tutoring?


Compare the top scores from the two questions. Pick a **threshold** between them: below it, treat the
question as out of scope. Around 0.35 to 0.45 usually works for this model, but check your own numbers.

In [9]:
THRESHOLD = 0.40   # adjust after looking at your scores

def relevant_chunks(question, k=3):
    return [(s, c) for s, c in retrieve(question, k) if s >= THRESHOLD]

print(len(relevant_chunks("Can I still study in the library at 11 at night during exams?")), "relevant")
print(len(relevant_chunks("Who won the hockey game last night?")), "relevant")

2 relevant
0 relevant


**Checkpoint:** the library question should keep at least one chunk; the hockey question should keep none.

## Step 6: Generate the answer with an LLM

This is the **G** in RAG. The retrieved chunks go into the prompt along with clear instructions:

- answer **only** from the context,
- say so when the context does not contain the answer,
- keep it short.

We use **Qwen2.5-0.5B-Instruct**, a small open-source model that runs free on your own laptop's CPU through
Hugging Face `transformers`. No account, no API key and no data leaves your machine. The first run downloads it
(about 1 GB), so start this cell early.

Small models are weaker than large commercial ones. That is part of the lesson: with good retrieval and a clear
prompt, even a 0.5-billion-parameter model gives accurate campus answers, because the facts come from the
context, not from the model's memory.

In [10]:
from transformers import pipeline

llm = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")

SYSTEM_PROMPT = (
    "You are the campus help assistant for Northbridge College. "
    "Answer the student's question using ONLY the FAQ context provided. "
    "If the context does not contain the answer, say you don't know and suggest "
    "contacting Student Services. Answer in one to three sentences."
)

def build_prompt(question, context_chunks):
    context = "\n\n".join(chunk for _, chunk in context_chunks)
    return f"<context>\n{context}\n</context>\n\nQuestion: {question}"

def generate(question, context_chunks):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_prompt(question, context_chunks)},
    ]
    output = llm(messages, max_new_tokens=150, do_sample=False)
    return output[0]["generated_text"][-1]["content"].strip()

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 371.58it/s]


**See what the model receives.** Print the prompt for the library question before sending it.

In [11]:
q = "Can I still study in the library at 11 at night during exams?"
print(build_prompt(q, relevant_chunks(q)))

<context>
Q: When is the library open?
A: The main library is open 8 AM to 10 PM Monday to Thursday, 8 AM to 6 PM on Friday, and 10 AM to 4 PM on weekends. During exam weeks it stays open until midnight.

Q: How do I book a study room?
A: Study rooms are booked through the library portal up to 7 days in advance. Each booking is 2 hours, and you can hold one booking at a time.
</context>

Question: Can I still study in the library at 11 at night during exams?


**Checkpoint:** the prompt should contain the library FAQ entry inside the `<context>` tags, followed by the question.

## Step 7: Put it together

`ask()` runs the full pipeline. If nothing passes the threshold, it answers "I don't know" **without
calling the LLM at all**. That is cheaper, faster and removes the chance of a made-up answer.

In [12]:
def ask(question):
    context = relevant_chunks(question)
    if not context:
        return "Sorry, I don't know that one. Please contact Student Services."
    return generate(question, context)

for q in ["Can I still study in the library at 11 at night during exams?",
          "I can't log in to the Wi-Fi, what network do I use?",
          "Is tutoring for Python free?",
          "Who won the hockey game last night?"]:
    print("Q:", q)
    print("A:", ask(q))
    print("-" * 70)

Q: Can I still study in the library at 11 at night during exams?
A: No, you cannot study in the library from 11 PM to 1 AM during exam periods. This is because the library is closed during these times.
----------------------------------------------------------------------
Q: I can't log in to the Wi-Fi, what network do I use?
A: To connect to campus Wi-Fi, you should use NB-Secure as it is the default network name. However, if you need to switch networks or have specific requirements, you might want to check the available options on campus or contact Student Services for assistance.
----------------------------------------------------------------------
Q: Is tutoring for Python free?
A: Yes, Python is also available for free tutoring at the Learning Centre on Mondays from 10 AM to 4 PM in Building B, Room 120.
----------------------------------------------------------------------
Q: Who won the hockey game last night?
A: Sorry, I don't know that one. Please contact Student Services.
--

**Checkpoint:** the first three questions should be answered from the FAQ in the model's own words. The last should get the "I don't know" reply.

Answers take a few seconds each on a laptop CPU. That is normal for a model running locally.

## Step 8: Make it interactive with Streamlit

A Streamlit app runs as its **own Python script**, separate from this notebook, so the variables above do not
exist there. The next cell writes everything the app needs into `app.py`.

`@st.cache_resource` loads the embedding model, the index and the LLM **once**. Without it, Streamlit would rebuild them every time
the user types a question.

Run the cell, then in a terminal (in the same folder):

```
streamlit run app.py
```

Your browser opens the chatbot at `http://localhost:8501`. In Colab, Streamlit needs a tunnel, so this step is
easiest on your own machine.

In [13]:
%%writefile app.py
import faiss
import numpy as np
import streamlit as st
from sentence_transformers import SentenceTransformer
from transformers import pipeline

FAQS = [
    ("When is the library open?", "The main library is open 8 AM to 10 PM Monday to Thursday, 8 AM to 6 PM on Friday, and 10 AM to 4 PM on weekends. During exam weeks it stays open until midnight."),
    ("How do I book a study room?", "Study rooms are booked through the library portal up to 7 days in advance. Each booking is 2 hours, and you can hold one booking at a time."),
    ("Where do I get my student ID card?", "Student ID cards are issued at the Student Services desk in Building A, room 104. Bring a government photo ID and your acceptance letter."),
    ("How do I connect to campus Wi-Fi?", "Connect to the network called NB-Secure and sign in with your student email and password. Guests can use NB-Guest, which needs no password but is slower."),
    ("What should I do if I forget my password?", "Reset it at the IT self-service portal using your recovery email. If that fails, visit the IT Help Desk in Building C, room 210."),
    ("Where can I print?", "Printers are in the library and in Building C. Black-and-white pages cost 10 cents and colour pages cost 50 cents, charged to your student card."),
    ("Is there free tutoring?", "Yes. The Learning Centre offers free drop-in tutoring for math, writing and programming, Monday to Friday from 10 AM to 4 PM in Building B, room 120."),
    ("How do I see a counsellor?", "Counselling is free and confidential. Book an appointment online through Student Wellness or call the front desk. Same-day crisis appointments are available."),
    ("Where can I park?", "Students park in Lots 3 and 4 with a semester permit, which costs $180 and is bought online. Lots 1 and 2 are for staff and visitors."),
    ("What is the deadline to drop a course without a failing grade?", "You can drop a course without academic penalty until the end of week 8. After that, the course shows as a failing grade on your transcript."),
    ("Does the cafeteria have vegetarian options?", "Yes. The main cafeteria in Building A has a vegetarian and vegan station every weekday from 11 AM to 3 PM, and halal options are labelled at every counter."),
    ("How do I contact Accessibility Services?", "Email Accessibility Services or visit Building A, room 112. Register early in the term so accommodations such as extra exam time are in place before your first test."),
]
THRESHOLD = 0.40
SYSTEM_PROMPT = (
    "You are the campus help assistant for Northbridge College. "
    "Answer the student's question using ONLY the FAQ context provided. "
    "If the context does not contain the answer, say you don't know and suggest "
    "contacting Student Services. Answer in one to three sentences."
)

@st.cache_resource
def load():
    chunks = [f"Q: {q}\nA: {a}" for q, a in FAQS]
    model = SentenceTransformer("all-MiniLM-L6-v2")
    emb = model.encode(chunks, normalize_embeddings=True).astype("float32")
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    llm = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")
    return chunks, model, index, llm

def retrieve(question, chunks, model, index, k=3):
    q = model.encode([question], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    return [(float(s), chunks[i]) for s, i in zip(scores[0], ids[0]) if s >= THRESHOLD]

def generate(question, context, llm):
    prompt = "<context>\n" + "\n\n".join(c for _, c in context) + f"\n</context>\n\nQuestion: {question}"
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}]
    output = llm(messages, max_new_tokens=150, do_sample=False)
    return output[0]["generated_text"][-1]["content"].strip()

st.title("Campus FAQ Chatbot")
st.caption("Northbridge College (sample data) - answers come only from the FAQ.")

with st.spinner("Loading models (first run downloads about 1 GB)..."):
    chunks, model, index, llm = load()
question = st.text_input("Ask a question about campus:")

if question:
    context = retrieve(question, chunks, model, index)
    if not context:
        st.warning("Sorry, I don't know that one. Please contact Student Services.")
    else:
        with st.spinner("Thinking..."):
            st.write(generate(question, context, llm))
        with st.expander("Sources used"):
            for score, chunk in context:
                st.markdown(f"**{score:.2f}**  \n{chunk}")


Writing app.py


## Step 9: Reflection

Answer each in two or three sentences.

1. **How does the chatbot "understand" the question?** It never reads words the way you do. What is it actually comparing?
2. **What happens if the user asks something not in the FAQ?** What would happen if you removed the threshold, and why is that dangerous for a campus help bot?
3. **Why did we keep each question and answer in the same chunk?** Try the line-by-line split from the original version and describe what goes wrong.
4. **How could you scale this** to the full student handbook (hundreds of pages) instead of 14 FAQ entries? Think about chunk size, the type of index, and keeping the data up to date.

## Stretch goals
- **Paraphrase test.** Write five questions that share no keywords with the FAQ. How many does retrieval still get right?
- **Change k.** Try `k=1` and `k=5`. When does more context help the LLM, and when does it add noise?
- **Show sources.** Make `ask()` return the answer plus the questions of the chunks it used, so a student can check where the answer came from.
- **Real data.** Copy ten real FAQ entries from your college website and rebuild the index.